# Sprint 3: Dynamic Feature Extraction Testing

This notebook demonstrates testing the dynamic feature extraction module with sample data.

## 1. Setup & Imports

In [4]:
import sys
import os

# Add the backend module path
backend_path = r"c:\Users\Jesun\Downloads\Project\Mentaiimage\canvas-app-main\canvas-app-main\backend\sprint-3-dynamic-features"
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Import our module
from dynamic_feature_extraction import (
    DynamicFeatureExtractor,
    DrawingSessionValidator,
    StrokeAnalyzer
)

print("✓ Imports successful")

✓ Imports successful


## 2. Create Test Session Data

Create a minimal but realistic drawing session for testing.

In [9]:
def create_test_session():
    """Create a valid test drawing session matching the exact schema."""
    return {
        "sessionId": "test-session-001",
        "taskStartTime": "2026-02-19T10:00:00Z",
        "taskEndTime": "2026-02-19T10:00:05Z",
        "strokes": [
            # Stroke 1: diagonal line
            {
                "strokeId": "stroke-1",
                "startTime": "2026-02-19T10:00:00Z",
                "endTime": "2026-02-19T10:00:01Z",
                "points": [
                    {"x": 100, "y": 100, "timestamp": 1645174800000, "pressure": 0.8, "tiltX": 0, "tiltY": 0},
                    {"x": 110, "y": 105, "timestamp": 1645174800016, "pressure": 0.8, "tiltX": 0, "tiltY": 0},
                    {"x": 120, "y": 115, "timestamp": 1645174800032, "pressure": 0.8, "tiltX": 0, "tiltY": 0},
                    {"x": 130, "y": 130, "timestamp": 1645174800048, "pressure": 0.8, "tiltX": 0, "tiltY": 0},
                    {"x": 140, "y": 145, "timestamp": 1645174800064, "pressure": 0.8, "tiltX": 0, "tiltY": 0},
                ]
            },
            # Stroke 2: curved shape
            {
                "strokeId": "stroke-2",
                "startTime": "2026-02-19T10:00:01.5Z",
                "endTime": "2026-02-19T10:00:02.5Z",
                "points": [
                    {"x": 200, "y": 100, "timestamp": 1645174801500, "pressure": 0.7, "tiltX": 5, "tiltY": 5},
                    {"x": 220, "y": 110, "timestamp": 1645174801516, "pressure": 0.7, "tiltX": 5, "tiltY": 5},
                    {"x": 235, "y": 130, "timestamp": 1645174801532, "pressure": 0.7, "tiltX": 5, "tiltY": 5},
                    {"x": 240, "y": 155, "timestamp": 1645174801548, "pressure": 0.7, "tiltX": 5, "tiltY": 5},
                    {"x": 235, "y": 175, "timestamp": 1645174801564, "pressure": 0.7, "tiltX": 5, "tiltY": 5},
                ]
            }
        ],
        "deviceInfo": {"type": "stylus", "model": "test-device"}
    }

test_session = create_test_session()
print(f"Created test session with {len(test_session['strokes'])} strokes")
print(f"Total points: {sum(len(s['points']) for s in test_session['strokes'])}")

Created test session with 2 strokes
Total points: 10


## 3. Test Validation

In [10]:
# Test the validation directly
validator = DrawingSessionValidator()
validation_result = validator.validate_session(test_session)

print(f"Validation Result:")
print(f"  Is Valid: {validation_result.is_valid}")
print(f"  Errors: {validation_result.errors if validation_result.errors else 'None'}")
print(f"  Warnings: {validation_result.warnings if validation_result.warnings else 'None'}")
print(f"\nStatistics:")
for key, value in validation_result.stats.items():
    print(f"  {key}: {value}")

Validation Result:
  Is Valid: True
  Errors: None
  Warnings: None

Statistics:
  session_duration_ms: 5000.0
  total_strokes: 2
  total_points: 10
  avg_points_per_stroke: 5.0


## 4. Test Feature Extraction

In [11]:
# Extract all features
extractor = DynamicFeatureExtractor(test_session, validate=True)
features = extractor.extract_all_features()

print("Feature Extraction Complete!\n")
print("=" * 50)
print("SPEED METRICS")
print("=" * 50)
print(f"Average speed:     {features.avg_speed:.4f} px/ms")
print(f"Max speed:         {features.max_speed:.4f} px/ms")
print(f"Min speed:         {features.min_speed:.4f} px/ms")
print(f"Speed variance:    {features.speed_variance:.6f} (px/ms)²")
print(f"Speed profile len: {len(features.speed_profile)} segments")

print(f"\n{'=' * 50}")
print("ACCELERATION METRICS")
print("=" * 50)
print(f"Average acceleration: {features.avg_acceleration:.6f} px/ms²")
print(f"Max acceleration:     {features.max_acceleration:.6f} px/ms²")
print(f"Acceleration variance: {features.acceleration_variance:.8f} (px/ms²)²")

print(f"\n{'=' * 50}")
print("TIMING METRICS")
print("=" * 50)
print(f"Total drawing time: {features.total_drawing_time:.1f} ms")
print(f"Total pause time:   {features.total_pause_time:.1f} ms")
print(f"Stroke durations:   {features.stroke_durations}")

print(f"\n{'=' * 50}")
print("PAUSE/RHYTHM METRICS")
print("=" * 50)
print(f"Number of pauses:         {features.number_of_pauses}")
print(f"Avg pause duration:       {features.avg_pause_duration:.1f} ms")
print(f"Max pause duration:       {features.max_pause_duration:.1f} ms")
print(f"Avg inter-stroke interval: {features.avg_inter_stroke_interval:.1f} ms")
print(f"Stroke frequency:         {features.stroke_frequency:.4f} strokes/sec")

print(f"\n{'=' * 50}")
print("DIRECTION & CURVATURE")
print("=" * 50)
print(f"Direction changes: {features.direction_changes}")
print(f"Avg curvature:     {features.avg_curvature:.6f} rad/pixel")
print(f"Max curvature:     {features.max_curvature:.6f} rad/pixel")

print(f"\n{'=' * 50}")
print("OVERALL STATISTICS")
print("=" * 50)
print(f"Total strokes:        {features.total_strokes}")
print(f"Total points:         {features.total_points}")
print(f"Avg points/stroke:    {features.avg_points_per_stroke:.1f}")
print(f"Drawing area:         {features.drawing_area:.1f} pixels²")

Feature Extraction Complete!

SPEED METRICS
Average speed:     1.2098 px/ms
Max speed:         1.5934 px/ms
Min speed:         0.6988 px/ms
Speed variance:    0.086773 (px/ms)²
Speed profile len: 8 segments

ACCELERATION METRICS
Average acceleration: 0.009675 px/ms²
Max acceleration:     0.019061 px/ms²
Acceleration variance: 0.00012856 (px/ms²)²

TIMING METRICS
Total drawing time: 2000.0 ms
Total pause time:   500.0 ms
Stroke durations:   [1000.0, 1000.0]

PAUSE/RHYTHM METRICS
Number of pauses:         1
Avg pause duration:       500.0 ms
Max pause duration:       500.0 ms
Avg inter-stroke interval: 500.0 ms
Stroke frequency:         0.4000 strokes/sec

DIRECTION & CURVATURE
Direction changes: 0
Avg curvature:     0.015687 rad/pixel
Max curvature:     0.025412 rad/pixel

OVERALL STATISTICS
Total strokes:        2
Total points:         10
Avg points/stroke:    5.0
Drawing area:         10500.0 pixels²


## 5. Test Individual Stroke Analysis

In [12]:
# Analyze first stroke in detail
stroke = test_session['strokes'][0]

print(f"Analyzing Stroke 1: {stroke['strokeId']}\n")

# Low-level calculations
distances = StrokeAnalyzer.calculate_distances(stroke)
time_deltas = StrokeAnalyzer.calculate_time_deltas(stroke)
velocities, avg_vel, max_vel, min_vel = StrokeAnalyzer.calculate_velocity(stroke)
accels, avg_accel, max_accel = StrokeAnalyzer.calculate_acceleration(stroke)
curvatures, avg_curv, max_curv = StrokeAnalyzer.calculate_curvature(stroke)
dir_changes = StrokeAnalyzer.count_direction_changes(stroke, angle_threshold=30)

print("Distances (pixels):", np.round(distances, 2))
print("Time deltas (ms):", time_deltas)
print(f"\nVelocities (px/ms): {np.round(velocities, 3)}")
print(f"  Avg: {avg_vel:.4f}, Max: {max_vel:.4f}, Min: {min_vel:.4f}")
print(f"\nAccelerations (px/ms²): {np.round(accels, 4)}")
print(f"  Avg: {avg_accel:.6f}, Max: {max_accel:.6f}")
print(f"\nCurvatures (rad/px): {np.round(curvatures, 4)}")
print(f"  Avg: {avg_curv:.6f}, Max: {max_curv:.6f}")
print(f"\nDirection changes (>30°): {dir_changes}")

Analyzing Stroke 1: stroke-1

Distances (pixels): [11.18 14.14 18.03 18.03]
Time deltas (ms): [16 16 16 16]

Velocities (px/ms): [0.699 0.884 1.127 1.127]
  Avg: 0.9590, Max: 1.1267, Min: 0.6988

Accelerations (px/ms²): [0.0116 0.0152 0.    ]
  Avg: 0.008916, Max: 0.015178

Curvatures (rad/px): [0.0254 0.0123 0.    ]
  Avg: 0.012561, Max: 0.025412

Direction changes (>30°): 0


## 6. Test DataFrame Export

In [19]:
# Convert to DataFrame
df = extractor.to_dataframe()

print("DataFrame shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few values:")
print(df.iloc[0, :5])

DataFrame shape: (1, 27)

Column names:
['avg_speed', 'max_speed', 'min_speed', 'speed_variance', 'speed_profile', 'avg_acceleration', 'max_acceleration', 'acceleration_variance', 'acceleration_profile', 'total_drawing_time', 'total_pause_time', 'stroke_durations', 'number_of_pauses', 'avg_pause_duration', 'max_pause_duration', 'pause_durations', 'inter_stroke_interval', 'avg_inter_stroke_interval', 'stroke_frequency', 'direction_changes', 'avg_curvature', 'max_curvature', 'curvature_profile', 'total_strokes', 'total_points', 'avg_points_per_stroke', 'drawing_area']

First few values:
avg_speed                                                   1.20976
max_speed                                                  1.593444
min_speed                                                  0.698771
speed_variance                                             0.086773
speed_profile     [0.6987712429686843, 0.8838834764831844, 1.126...
Name: 0, dtype: object


## 7. Test with Invalid Data (Error Handling)

In [20]:
# Test with missing fields
incomplete_session = {
    "sessionId": "bad-session",
    "taskStartTime": "2026-02-19T10:00:00Z",
    # Missing taskEndTime
    "strokes": []
}

print("Testing with incomplete session...")
validator = DrawingSessionValidator()
result = validator.validate_session(incomplete_session)

print(f"Is valid: {result.is_valid}")
print(f"Errors:")
for error in result.errors:
    print(f"  - {error}")

Testing with incomplete session...
Is valid: False
Errors:
  - Missing required field: deviceInfo


## 8. Test with Multiple Sessions (Batch)

In [21]:
# Create multiple test sessions
sessions = [create_test_session() for _ in range(3)]

# Update IDs to be unique
for i, session in enumerate(sessions):
    session['sessionId'] = f'test-session-{i:03d}'

# Extract features from all
all_features = []
for session in sessions:
    try:
        extractor = DynamicFeatureExtractor(session, validate=True)
        features = extractor.extract_all_features()
        all_features.append({
            'session_id': session['sessionId'],
            'avg_speed': features.avg_speed,
            'max_speed': features.max_speed,
            'avg_acceleration': features.avg_acceleration,
            'total_strokes': features.total_strokes,
            'total_points': features.total_points,
            'number_of_pauses': features.number_of_pauses,
            'avg_curvature': features.avg_curvature
        })
    except Exception as e:
        print(f"Error processing {session['sessionId']}: {e}")

# Create DataFrame
df_batch = pd.DataFrame(all_features)
print(f"\nExtracted features from {len(all_features)} sessions:\n")
print(df_batch.to_string(index=False))


Extracted features from 3 sessions:

      session_id  avg_speed  max_speed  avg_acceleration  total_strokes  total_points  number_of_pauses  avg_curvature
test-session-000    1.20976   1.593444          0.009675              2            10                 1       0.015687
test-session-001    1.20976   1.593444          0.009675              2            10                 1       0.015687
test-session-002    1.20976   1.593444          0.009675              2            10                 1       0.015687


## 9. Performance Test

In [22]:
import time

# Create a large session with many points
large_session = {
    "sessionId": "perf-test",
    "taskStartTime": "2026-02-19T10:00:00Z",
    "taskEndTime": "2026-02-19T10:00:10Z",
    "strokes": [
        {
            "strokeId": f"stroke-{i}",
            "startTime": f"2026-02-19T10:00:{i:02d}Z",
            "endTime": f"2026-02-19T10:00:{i+1:02d}Z",
            "points": [
                {
                    "x": 100 + j,
                    "y": 100 + j,
                    "timestamp": 1645174800000 + (i * 1000) + (j * 10),
                    "pressure": 0.8,
                    "tiltX": 0,
                    "tiltY": 0
                }
                for j in range(100)  # 100 points per stroke
            ]
        }
        for i in range(10)  # 10 strokes
    ],
    "deviceInfo": {"type": "stylus"}
}

total_points = sum(len(s['points']) for s in large_session['strokes'])
print(f"Performance test: {total_points} points across {len(large_session['strokes'])} strokes")

start = time.time()
extractor = DynamicFeatureExtractor(large_session, validate=True)
features = extractor.extract_all_features()
elapsed = time.time() - start

print(f"\nExtraction time: {elapsed*1000:.2f} ms")
print(f"Rate: {total_points/elapsed:.0f} points/second")

Performance test: 1000 points across 10 strokes

Extraction time: 13.20 ms
Rate: 75786 points/second


## 10. Summary

✅ **Tests Passed:**
- Validation works correctly
- Feature extraction produces reasonable metrics
- Individual stroke analysis works
- DataFrame export works
- Error handling for invalid data works
- Batch processing works
- Performance is acceptable

**Ready for integration with:**
- Static feature extraction (Aramide)
- EDA analysis (Deepak)
- Feature merging pipeline